In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$
\mathbf{y} = \frac{1}{1 + e^{-\mathbf{x}}}
$$

$$ y_i = \frac{1}{1 + e^{-x_i}} $$

$$
\frac{dy}{d\mathbf{x}} = y \odot (1 - y)
$$

$$
\frac{dy_i}{dx_i} = y_i(1-y_i)
$$

$$
J_f(\mathbf{x}) =
\begin{bmatrix}
    \frac{\partial y_1}{\partial x_1} & 0 & \cdots & 0 \\
    0 & \frac{\partial y_2}{\partial x_2} & \cdots & 0 \\
    \vdots & \vdots & \ddots & \vdots \\
    0 & 0 & \cdots & \frac{\partial y_n}{\partial x_n}
\end{bmatrix}
$$

$$ d \mathbf{y} = J_y(\mathbf{x}) \cdot d\mathbf{x} $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{d \mathbf{y}}, d\mathbf{y} \right \rangle =
\left \langle \frac{dF}{d\mathbf{x}}, d\mathbf{x} \right \rangle
$$

$$
\left \langle \frac{dF}{d \mathbf{y}}, J_y(\mathbf{x}) \cdot d\mathbf{x} \right \rangle =
\left \langle \frac{dF}{d\mathbf{x}}, d\mathbf{x} \right \rangle \implies
$$

$$ \frac{dF}{d\mathbf{x}} = J_y(\mathbf{x})^\top \, \frac{dF}{d\mathbf{y}} $$

$$ \frac{dF}{d\mathbf{x}} = \frac{dy}{d\mathbf{x}} \odot \frac{dF}{d\mathbf{y}} $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore


def _sigmoid_neg(x: tr.Tensor) -> tr.Tensor:
    """Numerically stable `sigmoid` for negative inputs."""

    exp = tr.exp(x)
    y = exp / (exp + 1)
    return y


def _sigmoid_pos(x: tr.Tensor) -> tr.Tensor:
    """Numerically stable `sigmoid` for positive inputs."""

    y = 1 / (1 + tr.exp(-x))
    return y


def sigmoid(x: tr.Tensor) -> tr.Tensor:
    """Numerically stable `sigmoid` function."""

    x = tr.as_tensor(x, dtype=tr.float32)

    mask = (x >= 0)
    y = tr.empty_like(x)
    y[mask] = _sigmoid_pos(x[mask])
    y[~mask] = _sigmoid_neg(x[~mask])
    return y


class SigmoidFunction(autograd.Function):
    """Custom autograd function for the `sigmoid`."""

    @staticmethod
    def forward(ctx, x: tr.Tensor) -> tr.Tensor:
        y = sigmoid(x)  
        ctx.save_for_backward(y)
        return y
    
    @staticmethod
    def backward(ctx, dF_df: tr.Tensor) -> tuple[tr.Tensor, ]:
        (y, ) = ctx.saved_tensors

        df_dx = y * (1 - y)
        assert df_dx.shape == y.shape

        dF_dx = df_dx * dF_df
        assert dF_dx.shape == y.shape

        return (dF_dx, )
    

class Sigmoid(nn.Module):
    """Custom module for the `sigmoid`."""
    
    def __init__(self) -> None:
        super().__init__()

    def forward(self, x: tr.Tensor) -> tr.Tensor:
        return SigmoidFunction.apply(x)

In [ ]:
# Unit tests

def test_basic() -> None:
    assert sigmoid(1000) == approx(1.0)
    assert sigmoid(-1000) == approx(0.0)
    assert sigmoid(0) == approx(0.5)
    assert sigmoid(1) == approx(0.731, atol=0.001, rtol=0)
    assert sigmoid(-1) == approx(0.269, atol=0.001, rtol=0)


def test_gradcheck(samples) -> None:
    def func(x: tr.Tensor) -> tr.Tensor:
        return SigmoidFunction.apply(x)

    x = tr.randn(samples, dtype=tr.float64, requires_grad=True)
    assert autograd.gradcheck(func, (x, ), eps=0.001, atol=0.001)


def test_compare(samples) -> None:
    x = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    y1 = Sigmoid()(x1)
    F1 = y1.sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    y2 = tr.sigmoid(x2)
    F2 = y2.sum()
    F2.backward()

    assert y1 == approx(y2)
    assert x1.grad == approx(x2.grad)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)